# Netztransparenz API – aFRR/mFRR Activated Volumes

This notebook demonstrates the raw API calls for activated aFRR/mFRR and documents the response structure.
It then inspects how to aggregate across TSOs and compute total activated volumes.


## Columns of interest
- afrr_activated_mw_pos
- afrr_activated_mw_neg
- mfrr_activated_mw_pos
- mfrr_activated_mw_neg
- activated_volume_pos_mw (afrr + mfrr)
- activated_volume_neg_mw (afrr + mfrr)


In [1]:
import os
import io
import requests
import pandas as pd
import polars as pl
from datetime import datetime
from zoneinfo import ZoneInfo


## Configuration
Set a **short** window to inspect raw responses. Adjust if needed.


In [2]:
START_UTC = '2025-11-01T00:00:00Z'
END_UTC   = '2025-11-03T00:00:00Z'

# Use NETZTRANSPARENZ_TOKEN or client credentials
TOKEN = os.getenv('NETZTRANSPARENZ_TOKEN')
CLIENT_ID = os.getenv('NETZTRANSPARENZ_CLIENT_ID')
CLIENT_SECRET = os.getenv('NETZTRANSPARENZ_CLIENT_SECRET')


In [3]:
def _ensure_bearer(token: str) -> str:
    return token if token.startswith('Bearer ') else f'Bearer {token}'

def _fetch_token_from_client_credentials(client_id: str, client_secret: str) -> str:
    url = 'https://identity.netztransparenz.de/users/connect/token'
    payload = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret,
    }
    headers = {'Content-Type': 'application/x-www-form-urlencoded'}
    resp = requests.post(url, data=payload, headers=headers, timeout=60)
    resp.raise_for_status()
    token = resp.json().get('access_token')
    if not token:
        raise RuntimeError('Token response missing access_token')
    return _ensure_bearer(token)

def get_token() -> str:
    if TOKEN:
        return _ensure_bearer(TOKEN)
    if CLIENT_ID and CLIENT_SECRET:
        return _fetch_token_from_client_credentials(CLIENT_ID, CLIENT_SECRET)
    raise RuntimeError('No token provided. Set NETZTRANSPARENZ_TOKEN or CLIENT_ID/CLIENT_SECRET.')


In [4]:
def to_berlin(ts_utc: str) -> str:
    dt = datetime.fromisoformat(ts_utc.replace('Z', '+00:00'))
    return dt.astimezone(ZoneInfo('Europe/Berlin')).strftime('%Y-%m-%dT%H:%M:%S')

def fetch_csv(url: str, token: str) -> str:
    resp = requests.get(url, headers={'Authorization': token}, timeout=60)
    resp.raise_for_status()
    return resp.text

def read_csv(text: str) -> pl.DataFrame:
    cleaned = text.lstrip('\ufeff').lstrip('ï»¿').replace('\r\n', '\n').replace('\r', '\n')
    return pl.read_csv(io.BytesIO(cleaned.encode('utf-8')), separator=';', infer_schema_length=2000, quote_char=None)


## Raw API calls
We fetch the aFRR and mFRR activated volumes for the same window.


In [5]:
token = get_token()
base = 'https://ds.netztransparenz.de/api/v1/data'

start_local = to_berlin(START_UTC)
end_local = to_berlin(END_UTC)

url_afrr = f"{base}/NrvSaldo/AktivierteSRL/Qualitaetsgesichert/{start_local}/{end_local}"
url_mfrr = f"{base}/NrvSaldo/AktivierteMRL/Qualitaetsgesichert/{start_local}/{end_local}"

afrr_raw = read_csv(fetch_csv(url_afrr, token))
mfrr_raw = read_csv(fetch_csv(url_mfrr, token))

afrr_raw.head()


Datum,Zeitzone,von,bis,Datenkategorie,Datentyp,Einheit,50Hertz (Positiv),Amprion (Positiv),TenneT TSO (Positiv),TransnetBW (Positiv),Deutschland (Positiv),50Hertz (Negativ),Amprion (Negativ),TenneT TSO (Negativ),TransnetBW (Negativ),Deutschland (Negativ)
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""01.11.2025""","""UTC""","""01:00""","""01:15""","""Aktivierte SRL""","""Qualitätsgesichert""","""MW""","""28,540""","""6,160""","""32,468""","""7,032""","""74,200""","""0,396""","""0,000""","""0,184""","""0,060""","""0,640"""
"""01.11.2025""","""UTC""","""01:15""","""01:30""","""Aktivierte SRL""","""Qualitätsgesichert""","""MW""","""1,452""","""0,104""","""0,364""","""0,000""","""1,920""","""1,252""","""0,000""","""1,244""","""0,460""","""2,956"""
"""01.11.2025""","""UTC""","""01:30""","""01:45""","""Aktivierte SRL""","""Qualitätsgesichert""","""MW""","""0,152""","""2,092""","""24,196""","""12,836""","""39,276""","""0,264""","""0,000""","""0,120""","""0,036""","""0,420"""
"""01.11.2025""","""UTC""","""01:45""","""02:00""","""Aktivierte SRL""","""Qualitätsgesichert""","""MW""","""21,236""","""41,260""","""64,328""","""86,568""","""213,392""","""0,024""","""0,000""","""0,000""","""0,000""","""0,024"""
"""01.11.2025""","""UTC""","""02:00""","""02:15""","""Aktivierte SRL""","""Qualitätsgesichert""","""MW""","""48,892""","""33,920""","""45,404""","""97,792""","""226,008""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000"""


In [6]:
mfrr_raw.head()


Datum,Zeitzone,von,bis,Datenkategorie,Datentyp,Einheit,50Hertz (Positiv),Amprion (Positiv),TenneT TSO (Positiv),TransnetBW (Positiv),Deutschland (Positiv),50Hertz (Negativ),Amprion (Negativ),TenneT TSO (Negativ),TransnetBW (Negativ),Deutschland (Negativ)
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""01.11.2025""","""UTC""","""01:00""","""01:15""","""Aktivierte MRL""","""Qualitätsgesichert""","""MW""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000"""
"""01.11.2025""","""UTC""","""01:15""","""01:30""","""Aktivierte MRL""","""Qualitätsgesichert""","""MW""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000"""
"""01.11.2025""","""UTC""","""01:30""","""01:45""","""Aktivierte MRL""","""Qualitätsgesichert""","""MW""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000"""
"""01.11.2025""","""UTC""","""01:45""","""02:00""","""Aktivierte MRL""","""Qualitätsgesichert""","""MW""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000"""
"""01.11.2025""","""UTC""","""02:00""","""02:15""","""Aktivierte MRL""","""Qualitätsgesichert""","""MW""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000""","""0,000"""


## Inspect response columns (TSO breakdown)
We expect 4 TSO columns for POS/NEG each. The goal is to **sum** across TSOs per timestamp.


In [7]:
afrr_raw.columns


['Datum',
 'Zeitzone',
 'von',
 'bis',
 'Datenkategorie',
 'Datentyp',
 'Einheit',
 '50Hertz (Positiv)',
 'Amprion (Positiv)',
 'TenneT TSO (Positiv)',
 'TransnetBW (Positiv)',
 'Deutschland (Positiv)',
 '50Hertz (Negativ)',
 'Amprion (Negativ)',
 'TenneT TSO (Negativ)',
 'TransnetBW (Negativ)',
 'Deutschland (Negativ)']

In [8]:
mfrr_raw.columns


['Datum',
 'Zeitzone',
 'von',
 'bis',
 'Datenkategorie',
 'Datentyp',
 'Einheit',
 '50Hertz (Positiv)',
 'Amprion (Positiv)',
 'TenneT TSO (Positiv)',
 'TransnetBW (Positiv)',
 'Deutschland (Positiv)',
 '50Hertz (Negativ)',
 'Amprion (Negativ)',
 'TenneT TSO (Negativ)',
 'TransnetBW (Negativ)',
 'Deutschland (Negativ)']

## Aggregation logic (sum across TSOs)
We sum all columns containing 'positiv' for POS and 'negativ' for NEG, excluding meta columns.


In [9]:
def normalize_cols(df: pl.DataFrame) -> pl.DataFrame:
    mapping = {c: c.strip() for c in df.columns}
    df = df.rename(mapping)
    mapping_lower = {c: c.lower() for c in df.columns}
    return df.rename(mapping_lower)

def tidy_activation(df: pl.DataFrame, prefix: str) -> pl.DataFrame:
    df = normalize_cols(df)
    date_col = 'datum' if 'datum' in df.columns else df.columns[0]
    time_col = 'von' if 'von' in df.columns else df.columns[1]
    ts = (
        pl.concat_str([pl.col(date_col), pl.lit(' '), pl.col(time_col)])
        .str.to_datetime('%d.%m.%Y %H:%M', strict=False)
        .dt.replace_time_zone('UTC')
        .alias('timestamp_utc')
    )
    meta_cols = {date_col, time_col, 'zeitzone'}
    pos_cols = [c for c in df.columns if 'positiv' in c and c not in meta_cols]
    neg_cols = [c for c in df.columns if 'negativ' in c and c not in meta_cols]

    def clean_num(col: str) -> pl.Expr:
        return (
            pl.col(col)
            .cast(pl.Utf8)
            .str.replace('.', '')
            .str.replace(',', '.')
            .cast(pl.Float64, strict=False)
        )

    clean = df.with_columns([ts] + [clean_num(c).alias(c) for c in (pos_cols + neg_cols)])
    clean = clean.with_columns(
        pl.sum_horizontal([pl.col(c) for c in pos_cols]).alias(f'{prefix}_pos'),
        pl.sum_horizontal([pl.col(c) for c in neg_cols]).alias(f'{prefix}_neg'),
    ).select(['timestamp_utc', f'{prefix}_pos', f'{prefix}_neg'])
    return clean.group_by('timestamp_utc').agg(
        pl.col(f'{prefix}_pos').sum(),
        pl.col(f'{prefix}_neg').sum(),
    ).sort('timestamp_utc')

afrr = tidy_activation(afrr_raw, 'afrr_activated_mw')
mfrr = tidy_activation(mfrr_raw, 'mfrr_activated_mw')

afrr.head()


timestamp_utc,afrr_activated_mw_pos,afrr_activated_mw_neg
"datetime[μs, UTC]",f64,f64
2025-11-01 01:00:00 UTC,15.4,1.28
2025-11-01 01:15:00 UTC,1.84,1.912
2025-11-01 01:30:00 UTC,16.552,0.84
2025-11-01 01:45:00 UTC,26.784,0.048
2025-11-01 02:00:00 UTC,52.016,0.0


In [10]:
mfrr.head()


timestamp_utc,mfrr_activated_mw_pos,mfrr_activated_mw_neg
"datetime[μs, UTC]",f64,f64
2025-11-01 01:00:00 UTC,0.0,0.0
2025-11-01 01:15:00 UTC,0.0,0.0
2025-11-01 01:30:00 UTC,0.0,0.0
2025-11-01 01:45:00 UTC,0.0,0.0
2025-11-01 02:00:00 UTC,0.0,0.0


## Total activated volume (aFRR + mFRR)
We fill missing mFRR values with 0.0 before summation.


In [11]:
total = afrr.join(mfrr, on='timestamp_utc', how='full')
total = total.with_columns(
    (pl.col('afrr_activated_mw_pos') + pl.col('mfrr_activated_mw_pos').fill_null(0.0)).alias('activated_volume_pos_mw'),
    (pl.col('afrr_activated_mw_neg') + pl.col('mfrr_activated_mw_neg').fill_null(0.0)).alias('activated_volume_neg_mw'),
)
total.head()


timestamp_utc,afrr_activated_mw_pos,afrr_activated_mw_neg,timestamp_utc_right,mfrr_activated_mw_pos,mfrr_activated_mw_neg,activated_volume_pos_mw,activated_volume_neg_mw
"datetime[μs, UTC]",f64,f64,"datetime[μs, UTC]",f64,f64,f64,f64
2025-11-01 01:00:00 UTC,15.4,1.28,2025-11-01 01:00:00 UTC,0.0,0.0,15.4,1.28
2025-11-01 01:15:00 UTC,1.84,1.912,2025-11-01 01:15:00 UTC,0.0,0.0,1.84,1.912
2025-11-01 01:30:00 UTC,16.552,0.84,2025-11-01 01:30:00 UTC,0.0,0.0,16.552,0.84
2025-11-01 01:45:00 UTC,26.784,0.048,2025-11-01 01:45:00 UTC,0.0,0.0,26.784,0.048
2025-11-01 02:00:00 UTC,52.016,0.0,2025-11-01 02:00:00 UTC,0.0,0.0,52.016,0.0


## Data quality quick check
Nulls and zeros for the derived columns.


In [12]:
for col in ['afrr_activated_mw_pos','afrr_activated_mw_neg','mfrr_activated_mw_pos','mfrr_activated_mw_neg','activated_volume_pos_mw','activated_volume_neg_mw']:
    if col in total.columns:
        stats = total.select(
            pl.col(col).is_null().sum().alias('nulls'),
            (pl.col(col) == 0).sum().alias('zeros'),
            pl.col(col).min().alias('min'),
            pl.col(col).max().alias('max'),
            pl.col(col).mean().alias('mean'),
        ).to_dicts()[0]
        print(col, stats)


afrr_activated_mw_pos {'nulls': 0, 'zeros': 50, 'min': 0.0, 'max': 252.272, 'mean': 26.301041666666663}
afrr_activated_mw_neg {'nulls': 0, 'zeros': 74, 'min': 0.0, 'max': 173.48, 'mean': 16.897916666666667}
mfrr_activated_mw_pos {'nulls': 0, 'zeros': 192, 'min': 0.0, 'max': 0.0, 'mean': 0.0}
mfrr_activated_mw_neg {'nulls': 0, 'zeros': 192, 'min': 0.0, 'max': 0.0, 'mean': 0.0}
activated_volume_pos_mw {'nulls': 0, 'zeros': 50, 'min': 0.0, 'max': 252.272, 'mean': 26.301041666666663}
activated_volume_neg_mw {'nulls': 0, 'zeros': 74, 'min': 0.0, 'max': 173.48, 'mean': 16.897916666666667}
